# 09 - Figures and Results Summary
**Addresses comment 6 (figures/tables/consistency).**  All figures are regenerated from the *final* checkpoints of this protocol, at 300 dpi (PNG + vector PDF), with a **provenance JSON** next to each (checkpoint, split, image id, seed, the exact numbers shown). Qualitative examples are chosen by a **fixed rule, not by eye**: the median-PSNR and the worst-PSNR *test* image of the baseline, seed = first finished seed.

The last cell writes `RESULTS_SUMMARY.md` - every number in the manuscript must be copied from there (or the CSV/JSON files it lists), never typed.

In [3]:
import os, sys, json, warnings, importlib
PROJECT_ROOT = os.path.abspath("..")
if not os.path.isdir(os.path.join(PROJECT_ROOT, "notebooks")):
    PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
# Prefer a configured data root only when it actually contains the paired folders.
def _has_paired_data(root):
    if not root:
        return False
    return ((os.path.isdir(os.path.join(root, "train", "train", "GT")) and
             os.path.isdir(os.path.join(root, "train", "train", "NoisyLR"))) or
            (os.path.isdir(os.path.join(root, "GT")) and
             os.path.isdir(os.path.join(root, "NoisyLR"))))
if not _has_paired_data(os.environ.get("SEMICON_DATA")):
    os.environ["SEMICON_DATA"] = PROJECT_ROOT
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import config as C
C = importlib.reload(C)
# Print explicit results and data roots below
from utils.data import load_npy
from utils.experiment import load_results, load_per_image
from utils.analysis import finished_seeds, load_run_model, restore
from utils import figures as FG, metrics as MT
RUNS = os.path.join(PROJECT_ROOT, "results_kaggle")
DATA = PROJECT_ROOT
C.OUT = RUNS
C.RUNS = os.path.join(RUNS, "runs")
C.TABLES = os.path.join(RUNS, "tables")
C.FIGS = os.path.join(RUNS, "figures")
C.DATA_ROOT = DATA
print("RUNS =", RUNS)
print("DATA =", DATA)
if not os.path.isfile(os.path.join(RUNS, "split_manifest.json")):
    raise FileNotFoundError(f"Kaggle results split manifest not found under RUNS={RUNS}")
splits, man = C.get_splits(); dev = C.device()
have = [c for c in ["baseline", "global_gc", "film_control", "ldmh_loss_only", "full", "NAFNet-small", "Restormer-small", "SwinIR-small"] if finished_seeds(C.RUNS, c)]
sd = {c: finished_seeds(C.RUNS, c)[0] for c in have}; print("using seed", sd)
missing = [c for c in ["baseline", "full"] if c not in sd]
if missing:
    raise FileNotFoundError(f"Required runs missing from {C.RUNS}: {missing}. Train them in results_kaggle first.")

RUNS = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_kaggle
DATA = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow
using seed {'baseline': 0, 'global_gc': 0, 'film_control': 0, 'full': 0}


In [4]:
# Fig: training curves + component health of the SAME checkpoint that produced the tables
h = os.path.join(C.RUNS, "full", f"seed{sd['full']}", "history.csv")
FG.training_curves(h, C.FIGS, "fig_training_curves_full", "Full model (seed %d)" % sd["full"],
                   {"history": h, "checkpoint": h.replace("history.csv", "best.pth"), "note": "same run as Table (seed %d)" % sd["full"]})
FG.component_health(h, C.FIGS, "fig_component_health_full")
hb = os.path.join(C.RUNS, "baseline", f"seed{sd['baseline']}", "history.csv"); FG.training_curves(hb, C.FIGS, "fig_training_curves_baseline", "Baseline (seed %d)" % sd["baseline"])
print("saved training / component-health figures")

saved training / component-health figures


In [5]:
# Qualitative: fixed selection rule
test_gt = [load_npy(g) for g, _ in splits["test"]]; test_ny = [load_npy(n) for _, n in splits["test"]]
pi = load_per_image(C.RUNS, "baseline", "psnr").mean(0); order = np.argsort(pi)
picks = {"median": int(order[len(order) // 2]), "worst": int(order[0])}
for tag, k in picks.items():
    x = torch.from_numpy(test_ny[k])[None, None].to(dev)
    gt = test_gt[k]; arrs = [("Clean GT", gt), ("Input (bilinear x2)", np.clip(np.kron(test_ny[k], np.ones((2, 2))), 0, 1))]
    mix = None
    for c in have:
        m, cfg, pth = load_run_model(C.RUNS, c, sd[c], dev)
        with torch.no_grad(): out = m(x)
        pr = (out[0] if isinstance(out, tuple) else out).clamp(0, 1)[0, 0].cpu().numpy()
        arrs.append((f"{c}\n{MT.psnr(torch.from_numpy(pr)[None,None], torch.from_numpy(gt)[None,None]).item():.2f} dB", pr))
        if c == "full": mix = (out[1][0].cpu().numpy(), out[2].cpu().numpy(), pth)
    name_ = os.path.basename(splits["test"][k][0])
    FG.qualitative_grid(arrs, C.FIGS, f"fig_qualitative_{tag}", {"split": "test", "image": name_, "rule": f"{tag} PSNR of baseline", "seeds": sd})
    if mix is not None:
        FG.mixture_maps(test_ny[k], mix[0], mix[1], C.FIGS, f"fig_mixture_maps_{tag}", {"split": "test", "image": name_, "checkpoint": mix[2], "seed": sd["full"]})
    print(tag, "->", name_)

median -> 001003.npy
worst -> 002639.npy


In [6]:
# provenance audit: every figure must have a JSON sidecar
import glob
figs = sorted(glob.glob(os.path.join(C.FIGS, "*.png")))
audit = [(os.path.basename(f), os.path.exists(f[:-4] + ".json")) for f in figs]
print(pd.DataFrame(audit, columns=["figure", "has_provenance_json"]).to_string(index=False))
assert all(a for _, a in audit), "figure without provenance sidecar"
for f in sorted(glob.glob(os.path.join(C.FIGS, "fig_mixture_maps_*.json"))):
    j = json.load(open(f)); print(os.path.basename(f), "betas:", [round(b, 3) for b in j["betas"]], "dominant fractions:", [round(x, 3) for x in j["dominant_fraction"]])
print("\nThe betas and dominant fractions printed above are the ONLY values that may be quoted for the mixture-map figure.")

                               figure  has_provenance_json
         fig_ablation_test_hp_err.png                 True
           fig_ablation_test_psnr.png                 True
           fig_ablation_test_ssim.png                 True
fig_beta_vs_structure_hetero_null.png                 True
   fig_beta_vs_structure_observed.png                 True
        fig_component_health_full.png                 True
           fig_data_contact_sheet.png                 True
                fig_distributions.png                 True
                fig_external_psnr.png                 True
          fig_mixture_maps_median.png                 True
           fig_mixture_maps_worst.png                 True
             fig_model_comparison.png                 True
           fig_noise_vs_intensity.png                 True
                        fig_pairs.png                 True
          fig_patch_beta_vs_nulls.png                 True
           fig_qualitative_median.png                 Tr

In [7]:
# RESULTS_SUMMARY.md : one file to copy numbers from
lines = ["# Results summary (auto-generated; copy numbers from here)\n", f"mode={C.MODE} epochs={C.EPOCHS} seeds={C.SEEDS}\n"]
for f in sorted(glob.glob(os.path.join(C.TABLES, "*.csv"))):
    if "long" in f or "per_image" in f or "all_runs" in f: continue
    d = pd.read_csv(f).round(4)
    try: body = d.to_markdown(index=False)
    except Exception: body = "```\n" + d.to_string(index=False) + "\n```"
    lines += [f"\n## {os.path.basename(f)}\n", body, "\n"]
for f in sorted(glob.glob(os.path.join(C.TABLES, "*.json"))):
    lines += [f"\n## {os.path.basename(f)}\n", "```", json.dumps(json.load(open(f)), indent=1, default=str)[:3000], "```\n"]
open(os.path.join(C.OUT, "RESULTS_SUMMARY.md"), "w").write("\n".join(lines)); print("wrote", os.path.join(C.OUT, "RESULTS_SUMMARY.md"))

wrote d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_kaggle\RESULTS_SUMMARY.md
